# Pseudolabels Preprocessing Pipeline

Walks through each stage of `preprocess_pseudolabels.py` on a single image,
with full-resolution visualisation at every step.

This pipeline includes **Gaussian rolling-ball background subtraction** and
**channel-role-aware** parameters to improve thresholding-based pseudolabel
quality. For the simpler SSL training pipeline (no bg subtraction), see
`preprocessing_pipeline_test.ipynb`.

**Pipeline:**
1. Load raw volume (C, Z, Y, X)
2. Maximum Intensity Projection along Z → (C, Y, X)
3. Per-channel background subtraction (Gaussian high-pass, role-aware)
4. Per-channel percentile normalisation → [0, 1]
5. Tiling into patches
6. Reconstruct tiled grid for visual sanity check
7. Side-by-side: with vs without background subtraction

---
## 0 — Imports & helpers

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, '..')

import gc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from utils_data.preprocess_pseudolabels import (
    load_image,
    maximum_intensity_projection,
    best_z_slice,
    denoise_mip,
    normalize_percentile,
    extract_patches,
    DEFAULT_CHANNEL_ROLES,
    ROLE_DEFAULTS,
)

print('Imports OK')

In [ ]:
# ── Visualisation helpers ────────────────────────────────────────────

def _black_to(rgb):
    """Colormap from black to the given RGB colour."""
    return LinearSegmentedColormap.from_list('', [(0, 0, 0), rgb])

CHANNEL_CMAPS = [
    _black_to((1, 0, 0)),   # ch0 — red
    _black_to((0, 1, 0)),   # ch1 — green
    _black_to((0, 0, 1)),   # ch2 — blue
]


def show_channels(image, title='', vmax_pct=99.5):
    """Show each channel of a (C, H, W) image side by side."""
    C = image.shape[0]
    fig, axes = plt.subplots(1, C, figsize=(6 * C, 6))
    if C == 1:
        axes = [axes]
    for c, ax in enumerate(axes):
        ch = image[c]
        vmax = np.percentile(ch, vmax_pct) if ch.max() > 0 else 1
        cmap = CHANNEL_CMAPS[c % len(CHANNEL_CMAPS)]
        ax.imshow(ch, cmap=cmap, vmin=0, vmax=vmax)
        ax.set_title(f'Ch {c}  min={ch.min():.3g}  max={ch.max():.3g}')
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

---
## 1 — Load raw volume

In [ ]:
# ── Point this at any raw microscopy file ──
IMAGE_PATH = '/run/media/anokhver/Data/Veronika/ctu/Microscopy/Microscopy/20251030/2_3_PSYHARMIN_2_Multichannel Z-Stack_20251030_67.vsi'
#IMAGE_PATH = '/run/media/anokhver/Data/Veronika/ctu/Microscopy/Microscopy/20251030/1_8_KONTROLA_1_Multichannel Z-Stack_20251030_118.vsi'

from pathlib import Path
volume = load_image(Path(IMAGE_PATH))   # (C, Z, Y, X)
C, Z, H, W = volume.shape
print(f'Raw volume: C={C}, Z={Z}, H={H}, W={W}, dtype={volume.dtype}')

In [ ]:
best_z = best_z_slice(volume)
mid_z = Z // 2
print(f'Middle z-slice: {mid_z},  brightest z-slice: {best_z}')

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, z_idx, label in [(axes[0], mid_z, 'Middle'), (axes[1], best_z, 'Brightest')]:
    composite = volume[:, z_idx, :, :].astype(np.float32).max(axis=0)
    vmax = np.percentile(composite, 99.5) if composite.max() > 0 else 1
    ax.imshow(composite, cmap='gray', vmin=0, vmax=vmax)
    ax.set_title(f'{label} z={z_idx}')
    ax.axis('off')
plt.suptitle('Middle vs brightest z-slice (max across channels)', fontsize=14)
plt.tight_layout()
plt.show()
plt.close(fig)

show_channels(volume[:, best_z, :, :].astype(np.float32),
              title=f'Brightest z-slice {best_z}/{Z}')

---
## 2 — Maximum Intensity Projection (MIP)

In [ ]:
mip = maximum_intensity_projection(volume)   # (C, Y, X) float32
print(f'MIP shape: {mip.shape}, dtype={mip.dtype}')

del volume
gc.collect()

show_channels(mip, title='After MIP (raw intensities)')

In [ ]:
# Per-channel intensity histograms before background subtraction
fig, axes = plt.subplots(1, mip.shape[0], figsize=(6 * mip.shape[0], 4))
if mip.shape[0] == 1:
    axes = [axes]
for c, ax in enumerate(axes):
    vals = mip[c].ravel()
    ax.hist(vals, bins=256, color=['r', 'g', 'b'][c % 3], alpha=0.7)
    ax.set_title(f'Ch {c} histogram')
    ax.set_yscale('log')
    ax.set_xlabel('Intensity')
fig.suptitle('MIP intensity distributions (before background subtraction)', fontsize=14)
plt.tight_layout()
plt.show()
plt.close(fig)

---
## 3 — Per-channel background subtraction

Apply Gaussian high-pass background subtraction (SynBot ordering: MIP first,
then subtract background on the 2D projection). The structural channel is
skipped because its dense neurite staining would be erased by the
Gaussian background estimate.

Parameters are loaded from `ROLE_DEFAULTS` — each channel role
(`pre_synaptic`, `post_synaptic`, `structural`) has its own settings.

In [ ]:
# Per-channel background-subtraction settings from ROLE_DEFAULTS
CHANNEL_ROLES = DEFAULT_CHANNEL_ROLES
MEDIAN_SIZES = [ROLE_DEFAULTS[r]["median_size"] for r in CHANNEL_ROLES]
ROLLING_BALL_RADII = [ROLE_DEFAULTS[r]["rolling_ball_radius"] for r in CHANNEL_ROLES]

print(f'Channel roles: {CHANNEL_ROLES}')
print(f'Median filter sizes: {MEDIAN_SIZES}')
print(f'Rolling-ball radii:  {ROLLING_BALL_RADII}')

mip_denoised = denoise_mip(mip, median_size=MEDIAN_SIZES, rolling_ball_radius=ROLLING_BALL_RADII)
print(f'\nDenoised MIP shape: {mip_denoised.shape}, dtype={mip_denoised.dtype}')

In [ ]:
show_channels(mip_denoised, title='After background subtraction')

In [ ]:
# Per-channel histograms after background subtraction
fig, axes = plt.subplots(1, mip_denoised.shape[0], figsize=(6 * mip_denoised.shape[0], 4))
if mip_denoised.shape[0] == 1:
    axes = [axes]
for c, ax in enumerate(axes):
    vals = mip_denoised[c].ravel()
    ax.hist(vals, bins=256, color=['r', 'g', 'b'][c % 3], alpha=0.7)
    ax.set_title(f'Ch {c} histogram (after bg subtraction)')
    ax.set_yscale('log')
    ax.set_xlabel('Intensity')
fig.suptitle('Intensity distributions after background subtraction', fontsize=14)
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# Side-by-side: raw MIP vs denoised MIP per channel
C_img = mip.shape[0]
fig, axes = plt.subplots(2, C_img, figsize=(6 * C_img, 10))
if C_img == 1:
    axes = axes.reshape(-1, 1)
for c in range(C_img):
    vmax = np.percentile(mip[c], 99.5) if mip[c].max() > 0 else 1
    cmap = CHANNEL_CMAPS[c % len(CHANNEL_CMAPS)]
    axes[0, c].imshow(mip[c], cmap=cmap, vmin=0, vmax=vmax)
    axes[0, c].set_title(f'Ch {c} — raw MIP')
    axes[0, c].axis('off')
    axes[1, c].imshow(mip_denoised[c], cmap=cmap, vmin=0, vmax=vmax)
    axes[1, c].set_title(f'Ch {c} — after bg subtraction')
    axes[1, c].axis('off')
fig.suptitle('Raw MIP vs background-subtracted MIP (same intensity scale)', fontsize=14)
plt.tight_layout()
plt.show()
plt.close(fig)

---
## 4 — Per-channel percentile normalisation

In [ ]:
PLOW  = 1.0
PHIGH = 99.8

mip_norm = normalize_percentile(mip_denoised, plow=PLOW, phigh=PHIGH)
print('Normalised range per channel:')
for c in range(mip_norm.shape[0]):
    print(f'  Ch {c}: [{mip_norm[c].min():.4f}, {mip_norm[c].max():.4f}]')

# Also normalise the raw MIP for comparison in step 7
mip_norm_raw = normalize_percentile(mip, plow=PLOW, phigh=PHIGH)
del mip
gc.collect()

In [ ]:
show_channels(mip_norm, title=f'After percentile normalisation  p=[{PLOW}, {PHIGH}]')

In [ ]:
# Per-channel histograms after normalisation
fig, axes = plt.subplots(1, mip_norm.shape[0], figsize=(6 * mip_norm.shape[0], 4))
if mip_norm.shape[0] == 1:
    axes = [axes]
for c, ax in enumerate(axes):
    vals = mip_norm[c].ravel()
    ax.hist(vals, bins=256, color=['r', 'g', 'b'][c % 3], alpha=0.7)
    ax.set_title(f'Ch {c} histogram (normalised)')
    ax.set_yscale('log')
    ax.set_xlim(0, 1)
    ax.set_xlabel('Intensity [0\u20131]')
fig.suptitle('Normalised intensity distributions', fontsize=14)
plt.tight_layout()
plt.show()
plt.close(fig)

---
## 5 — Tiling into patches

In [ ]:
PATCH_SIZE = 128

patches = extract_patches(mip_norm, patch_size=PATCH_SIZE)
n_total = patches.shape[0]

C_img, H_img, W_img = mip_norm.shape
n_rows = H_img // PATCH_SIZE
n_cols = W_img // PATCH_SIZE

print(f'Patches: {patches.shape}  ({n_rows} rows \u00d7 {n_cols} cols = {n_total})')

In [ ]:
# Show a few example patches
n_show = min(8, n_total)
n_cols_show = n_show // 2

fig, axes = plt.subplots(2, n_cols_show, figsize=(3 * n_cols_show, 7))
for i in range(n_show):
    r, c_ax = divmod(i, n_cols_show)
    p = patches[i]
    axes[r, c_ax].imshow(p[0], cmap='gray', vmin=0, vmax=1)
    axes[r, c_ax].set_title(f'#{i}\nmean={p.mean():.3f}')
    axes[r, c_ax].axis('off')

fig.suptitle('Example patches \u2014 Ch 0', fontsize=13)
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# Distribution of patch mean intensities
means = np.array([patches[i].mean() for i in range(n_total)])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(means, bins=64, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Mean intensity')
ax.set_ylabel('Count')
ax.set_title(f'Patch mean-intensity distribution  ({n_total} patches)')
plt.tight_layout()
plt.show()
plt.close(fig)

---
## 6 — Reassemble tile grid (visual sanity check)

Reconstruct the full image from patches to verify tiling is correct.

In [ ]:
def reassemble_grid(patches, n_rows, n_cols, patch_size, channel=0):
    """Stitch patches back into a full-size image for a single channel."""
    H = n_rows * patch_size
    W = n_cols * patch_size
    canvas = np.zeros((H, W), dtype=np.float32)

    for idx in range(len(patches)):
        r = idx // n_cols
        c = idx % n_cols
        y0 = r * patch_size
        x0 = c * patch_size
        canvas[y0:y0 + patch_size, x0:x0 + patch_size] = patches[idx, channel]

    return canvas

In [ ]:
fig, axes = plt.subplots(1, C_img, figsize=(7 * C_img, 7))
if C_img == 1:
    axes = [axes]

for c, ax in enumerate(axes):
    grid = reassemble_grid(patches, n_rows, n_cols, PATCH_SIZE, channel=c)
    cmap = CHANNEL_CMAPS[c % len(CHANNEL_CMAPS)]
    ax.imshow(grid, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(f'Ch {c} \u2014 reassembled patches')
    ax.axis('off')

fig.suptitle(f'Patch grid  {n_rows}\u00d7{n_cols}  |  patch_size={PATCH_SIZE}',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
plt.close(fig)

---
## 7 — With vs without background subtraction

Compare the pseudolabels pipeline (with bg subtraction) against the
training pipeline (no bg subtraction) to see the effect of the
Gaussian high-pass on each channel.

In [ ]:
fig, axes = plt.subplots(C_img, 2, figsize=(14, 7 * C_img))
if C_img == 1:
    axes = axes.reshape(1, 2)

for c in range(C_img):
    cmap = CHANNEL_CMAPS[c % len(CHANNEL_CMAPS)]
    role = CHANNEL_ROLES[c] if c < len(CHANNEL_ROLES) else '?'

    # Without bg subtraction (training pipeline)
    axes[c, 0].imshow(mip_norm_raw[c], cmap=cmap, vmin=0, vmax=1)
    axes[c, 0].set_title(f'Ch {c} ({role}) \u2014 no bg subtraction')
    axes[c, 0].axis('off')

    # With bg subtraction (pseudolabels pipeline)
    axes[c, 1].imshow(mip_norm[c], cmap=cmap, vmin=0, vmax=1)
    axes[c, 1].set_title(f'Ch {c} ({role}) \u2014 with bg subtraction')
    axes[c, 1].axis('off')

fig.suptitle('Training pipeline (no bg sub)  vs  Pseudolabels pipeline (with bg sub)',
             fontsize=15, y=1.01)
plt.tight_layout()
plt.show()
plt.close(fig)